In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../Dataset/car-sales-extended-missing-data1.csv")
data

,Make,Colour,Odometer (KM),Doors,Price
0,Honda,White,35431.0,4.0,15323.0
1,BMW,Blue,192714.0,5.0,19943.0
2,Honda,White,84714.0,4.0,28343.0
3,Toyota,White,154365.0,4.0,13434.0
4,Nissan,Blue,181577.0,3.0,14043.0
...,...,...,...,...,...
995,Toyota,Black,35820.0,4.0,32042.0
996,NaN,White,155144.0,3.0,5716.0
997,Nissan,Blue,66604.0,4.0,31570.0
998,Honda,White,215883.0,4.0,4001.0


In [3]:
data.dtypes

Make                 str
Colour               str
Odometer (KM)    float64
Doors            float64
Price            float64
dtype: object

In [4]:
data.isna().sum()

Make             49
Colour           50
Odometer (KM)    50
Doors            50
Price            50
dtype: int64

Steps we want to do all in 1 cell
1. Fill missing data
2. Convert data to numbers
3. build a model on the data

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

#modeling
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV

#setup random seed
np.random.seed(42)

#import data and drop rows with missing values
data = pd.read_csv("../Dataset/car-sales-extended-missing-data1.csv")
data.dropna(subset=["Price"], inplace=True)

#define different features and transformer pipelines
categorial_features = ["Make","Colour"]
categorial_transformer = Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot" ,OneHotEncoder(handle_unknown="ignore"))
])

door_fet = ["Doors"]
door_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant",fill_value=4))
])

numeric_fet = ["Odometer (KM)"]
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean"))
])

#steup the preprocing steps(fill missing values then convert to numbers)
preprocessor = ColumnTransformer(
    transformers=[
        ("cat",categorial_transformer,categorial_features),
        ("door",door_transformer,door_fet),
        ("num",numeric_transformer,numeric_fet)
    ]
)

#create a preprocessing and modeling pipeline
model = Pipeline(steps=[("preprocessor",preprocessor),
                        ("model",RandomForestRegressor())])

#split data
x = data.drop("Price",axis=1)
y = data["Price"]
X_train, X_test , y_train , y_test = train_test_split(x,y, test_size=0.2)

#fitting the model and score
model.fit(X_train,y_train)
model.score(X_test,y_test)

0.22188417408787875